# 08 Test Suite Runner

Run the full pytest suite with progress bars. Default mode runs one pytest invocation per test node for granular progress.

In [ ]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


In [ ]:
import subprocess
import sys
import time
from pathlib import Path

try:
    from tqdm.auto import tqdm
except Exception:
    class tqdm:  # lightweight fallback when tqdm is unavailable
        def __init__(self, iterable, total=None, desc='', unit='it'):
            self.iterable = list(iterable)
            self.total = len(self.iterable) if total is None else total
            self.desc = desc
            self.unit = unit
            self.idx = 0
            print(f'{self.desc}: 0/{self.total} {self.unit}')
        def __iter__(self):
            for item in self.iterable:
                yield item
                self.idx += 1
                print(f'{self.desc}: {self.idx}/{self.total} {self.unit}', end='\r')
            print()
        def set_postfix(self, **kwargs):
            return None

TEST_DIR = REPO_ROOT / 'tests'
assert TEST_DIR.exists(), f'Missing tests directory: {TEST_DIR}'
print('Test directory:', TEST_DIR)

In [ ]:
def collect_test_nodeids(repo_root: Path) -> list[str]:
    cmd = [sys.executable, '-m', 'pytest', '--collect-only', '-q', str(repo_root / 'tests')]
    proc = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError('Collection failed:\n' + proc.stdout + '\n' + proc.stderr)
    nodeids = []
    for line in proc.stdout.splitlines():
        line = line.strip()
        if '::' in line and line.startswith('tests/'):
            nodeids.append(line)
    return nodeids

nodeids = collect_test_nodeids(REPO_ROOT)
print(f'Collected {len(nodeids)} test nodes')
nodeids[:10]

In [ ]:
results = []
start = time.time()
for nodeid in tqdm(nodeids, total=len(nodeids), desc='Pytest progress', unit='test'):
    cmd = [sys.executable, '-m', 'pytest', '-q', nodeid]
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True)
    passed = (proc.returncode == 0)
    results.append({
        'nodeid': nodeid,
        'status': 'PASS' if passed else 'FAIL',
        'returncode': proc.returncode,
        'stdout': proc.stdout,
        'stderr': proc.stderr,
    })
elapsed = time.time() - start
print(f'Completed in {elapsed:.2f} sec')

In [ ]:
import pandas as pd
res_df = pd.DataFrame(results)
display(res_df[['nodeid', 'status']])
print(res_df['status'].value_counts())

failed = res_df[res_df['status'] == 'FAIL']
if len(failed):
    print('\nFailing tests and outputs:')
    for row in failed.itertuples(index=False):
        print('\n' + '=' * 80)
        print(row.nodeid)
        print('-' * 80)
        print(row.stdout)
        if row.stderr:
            print('[stderr]')
            print(row.stderr)
    raise AssertionError(f'{len(failed)} tests failed')
else:
    print('All tests passed.')

In [ ]:
output_path = REPO_ROOT / 'outputs' / 'test_suite' / 'pytest_progress_results.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
res_df.to_csv(output_path, index=False)
print('Saved detailed results to', output_path)